# The case study, in a notebook

`case-study/` is a corpus of **thirty claims from one public filing** — a verified complaint for
forfeiture in *United States v. Approximately 127,271 BTC* (E.D.N.Y. 1:25-cv-05745) and its schedule
of property — and a verdict for each.

**What this notebook is.** A *view* of that corpus. The report beside it (`results/report.md`) states
the coverage split in counts; what it cannot show is **why** each verdict came out as it did, because
the record carries the verdict and the finding behind it lives in the run. This shows the finding:
the reason, the evidence it rests on, and the derivation the library composes from them.

**What it is not.** A rate, a conclusion about any person, or a second way to read the corpus. Every
label, value and detail below arrives on a document the library produced — this renders, it does not
re-derive.

**Reproducing it.** The live cells below run the engine against the chain, so this file is committed
*with its outputs* and has no drift guard — the one generated artefact in this repository that does
not. `make case-study-notebook` re-runs it. `SELECTION.md` names the provider and says why.


## Set up

The live cells read the chain, so they need a provider and the corpus. The corpus is gitignored: it is
what `make ingest` wrote from the filing, and `SELECTION.md` names it.


In [1]:
from __future__ import annotations

import tomllib
from pathlib import Path

from IPython.display import Markdown, display

from chainlens.ledger.derive import derive_finding
from chainlens.models.derive import DerivationNode
from chainlens.providers.registry import get_registry
from chainlens.verify.engine import VerificationEngine
from chainlens.verify.estimators import estimator_for
from chainlens.verify.records import ClaimRecord, load_record
from chainlens.verify.schema import Extraction
from chainlens.verify.verdicts import VerificationFinding


def _case_dir() -> Path:
    """The case-study directory, found from wherever the kernel started.

    A notebook kernel starts in the directory the notebook is in, so a relative path would make this
    depend on how it was launched. Walking up until the corpus is visible means the cell works from
    the repository root or from `case-study/`, and fails loudly anywhere else.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "case-study" / "claims").is_dir():
            return candidate / "case-study"
    raise SystemExit("no case-study/claims found here; run this from the repository root")


CASE = _case_dir()
CLAIMS, CORPUS = CASE / "claims", CASE / "corpus"
assert CORPUS.is_dir(), "the corpus is gitignored — run `make ingest` first (see SELECTION.md)"

# **Named, not the registry default.** `esplora-mempool` took seven seconds to return a single
# block height while this corpus was built, so a two-thousand-transaction walk timed out;
# blockstream is the same Esplora API and answers in one. `SELECTION.md` gives the same reason.
PROVIDER = "esplora-blockstream"
print("claims:", len(list(CLAIMS.glob("*.toml"))), "| provider:", PROVIDER)

claims: 30 | provider: esplora-blockstream


## The corpus

Each record is one claim the filing states, with the reading this study applies to it — whether the
sentence is a **holding** (a schedule row, a claim about a past state), a **movement** (a figure
spanning a period, so a total over many movements) or a **receipt** (a destination named and no
payer). `SELECTION.md` says why each is stated on the record rather than inferred at check time.


In [2]:
def classification(claim: dict) -> str:
    """Which of the filing's three forms a record states, read off the record."""
    if claim["type"] == "balance":
        return "holding"
    return "receipt" if claim.get("direction") == "in" else "movement"


rows = []
for path in sorted(CLAIMS.glob("*.toml")):
    raw = tomllib.loads(path.read_text(encoding="utf-8"))
    rows.append(
        {
            "id": raw["id"],
            "class": classification(raw["claim"]),
            "quote": " ".join(raw["quote"].split()),
            "verdict": raw["expected"]["verdict"],
            "gap": raw["expected"].get("kind", ""),
        }
    )

table = ["| id | what it is | the filing's words | verdict | gap |", "|---|---|---|---|---|"]
table += [
    f"| {r['id']} | {r['class']} | {r['quote'][:64]}… | `{r['verdict']}` | {r['gap']} |"
    for r in rows
]
display(Markdown("\n".join(table)))

| id | what it is | the filing's words | verdict | gap |
|---|---|---|---|---|
| 0001 | holding | (a) 20,452.85228 BTC at address 3Pja5FPK1wFB9LkWWJai8XYL1qjbqqT9… | `unresolved` | no_data |
| 0002 | holding | (b) 14,111.92546835 BTC at address 3FrM1He2ZDbsSKmYpEZQNGjFTLMgC… | `unresolved` | no_data |
| 0003 | holding | (c) 2,999.09118947 BTC at address 3B1u4PsuFzww1P8if5jYmitXxpMs2E… | `unresolved` | no_data |
| 0004 | holding | (d) 1,000.08105870 BTC at address 3JJ8b7voMPSPChHazdHkrZMqxC7Cb4… | `unresolved` | no_data |
| 0005 | holding | (e) 0.00736862 BTC at address 3PWNGS2357TnjRX7FpewqR3e3qsWwpFrJH… | `unresolved` | no_data |
| 0006 | holding | (f) 14,139.260 BTC at address 34Jpa4Eu3ApoPVUKNTN2WeuXVVq1jzxgPi… | `unresolved` | no_data |
| 0007 | holding | (g) 9,099.01146835 BTC at address 338uPVW8drux5gSemDS4gFLSGrSfAi… | `unresolved` | no_data |
| 0008 | holding | (h) 499.90936500 BTC at address 3J4sTPyD1g6KvNUSJxjwLs4iaPeDPqxU… | `unresolved` | no_data |
| 0009 | holding | (i) 3,000.09125022 BTC at address 33uEsaGLcF9H46Dvzx1kMnuMCQ13nd… | `unresolved` | no_data |
| 0010 | holding | (j) 9,500.99220072 BTC at address 3KabDvdetZXDHNm9HXowLc9SppiSXK… | `unresolved` | no_data |
| 0011 | holding | (k) 15,033.29416267 BTC at address 38Md7BghVmV7XUUT1Vt9CvVcc5ssM… | `unresolved` | no_data |
| 0012 | holding | (l) 0.02415042 BTC at address 3GaB3nRWA1PLc3XQkkbpVtFwYYZEuMxD4i… | `unresolved` | no_data |
| 0013 | holding | (m) 3,000.09118974 BTC at address 32i6n2vXhjvJg1vniURFy7A5VK6eG6… | `unresolved` | no_data |
| 0014 | holding | (n) 4,500.00841044 BTC at address 3HuUiXmKN3beQSoM97kWjK1fesWWJv… | `unresolved` | no_data |
| 0015 | holding | (o) 0.5084661 BTC at address 34MFtk9iMxYcUPZWXHfiGfqz4o7X3kpJbV… | `unresolved` | no_data |
| 0016 | holding | (p) 156.04996844 BTC at address 3LjTXe31gepN8nW3AZyKpyD2QwbtmfjN… | `unresolved` | no_data |
| 0017 | holding | (q) 2,700.44863780 BTC at address 3MHa8JJ3bu8j3x3iQHhqsrZvk1EjBQ… | `unresolved` | no_data |
| 0018 | holding | (r) 10,500.04293955 BTC at address 3AWpzKtkHfWsiv9RGXKA3Z8951Lef… | `unresolved` | no_data |
| 0019 | holding | (s) 4,500.00941044 BTC at address 34KYo7VdVr5CJ7m4hYhH9RpwqXhbsT… | `unresolved` | no_data |
| 0020 | holding | (t) 251.6000482 BTC at address 3DdFSGcXaP2rZ9CaL3tjnqRARvQ5K3VW4… | `unresolved` | no_data |
| 0021 | holding | (u) 212.5930613 BTC at address 39B6oSa58qNpFMGpuowtRHAYp3fM4ghXR… | `unresolved` | no_data |
| 0022 | holding | (v) 8,611.07446862 BTC at address 3NmHmQte2rP8pS54U3B8LPYQKkpG1p… | `unresolved` | no_data |
| 0023 | holding | (x) 1,500.01255361 BTC at address 389JrNcn8trYgYi2EtHi4X7bTCqtVb… | `unresolved` | no_data |
| 0024 | holding | (y) 1,500.00 BTC at address 339khCuymVi4FKbW9hCHkH3CQwdopXiTvA… | `unresolved` | no_data |
| 0025 | receipt | On or about and between May 31, 2019 and March 29, 2020, approxi… | `contradicted` |  |
| 0026 | movement | on or about and between July 18, 2019 and September 30, 2019, th… | `contradicted` |  |
| 0027 | movement | On or about and between October 15, 2019 and January 24, 2020, t… | `contradicted` |  |
| 0028 | movement | On or about and between January 25, 2020 and March 21, 2020, the… | `contradicted` |  |
| 0029 | movement | On or about and between October 15, 2019 and February 27, 2020, … | `contradicted` |  |
| 0030 | movement | On or about and between July 2, 2019 and May 9, 2020, approximat… | `contradicted` |  |

## The split

In [3]:
from collections import Counter

by_class = Counter((row["class"], row["verdict"]) for row in rows)
table = ["| what the filing states | verdict | claims |", "|---|---|---|"]
table += [
    f"| {kind} | `{verdict}` | {count} |" for (kind, verdict), count in sorted(by_class.items())
]
display(Markdown("\n".join(table)))
display(
    Markdown(
        "Nothing reaches `supported`. The library verifies **a single payment between two named "
        "addresses**; a filing states holdings, cluster-level flows and period totals. "
        "`results/report.md` states that in full."
    )
)

| what the filing states | verdict | claims |
|---|---|---|
| holding | `unresolved` | 24 |
| movement | `contradicted` | 5 |
| receipt | `contradicted` | 1 |

Nothing reaches `supported`. The library verifies **a single payment between two named addresses**; a filing states holdings, cluster-level flows and period totals. `results/report.md` states that in full.

## Why each verdict came out as it did

One claim per class, checked live. The tree below is the library's own derivation: its nodes carry
their labels and their facts already composed, and this prints them.

There is **no markdown rendering of a derivation in the library** — the only one is the web app's
React component — so the walk is here, and it re-derives nothing.


In [4]:
#: How many facts of one node to print, and how long one of them may be. **A display decision, and
#: it says what it dropped.** One claim here — a receipt, whose total sums every payer — carries
#: 1,578 transaction ids in a single node, which is a hundred and twenty thousand characters of
#: derivation and is not evidence a reader can check. The count is printed either way.
DETAIL_LIMIT, VALUE_LIMIT = 12, 88


def _value(value: object) -> str:
    if isinstance(value, (list, tuple)):
        head = ", ".join(map(str, value[:3]))
        return f"{len(value)} items: {head}…" if len(value) > 3 else head
    text = str(value)
    return text if len(text) <= VALUE_LIMIT else text[:VALUE_LIMIT] + "…"


def _facts(node: DerivationNode) -> list[str]:
    """The node's facts as `key: value` lines, capped at `DETAIL_LIMIT` of them."""
    return [f"`{entry.key}`: {_value(entry.value)}" for entry in node.detail[:DETAIL_LIMIT]]


def render(node: DerivationNode, depth: int = 0) -> list[str]:
    """One derivation node and its children, as markdown."""
    lines, indent = [], "  " * depth
    head = f"{indent}- **{node.kind.value}** — {node.label}"
    if node.value is not None:
        head += f" = {node.value:g} {node.unit or ''}".rstrip()
    if node.band is not None:
        head += f"  _({node.band.value})_"
    lines.append(head)
    if node.summary:
        lines.append(f"{indent}  {node.summary}")
    lines += [f"{indent}  - {fact}" for fact in _facts(node)]
    if len(node.detail) > DETAIL_LIMIT:
        dropped = len(node.detail) - DETAIL_LIMIT
        lines.append(f"{indent}  - … and {dropped} more fact(s), not printed")
    for child in node.children:
        lines += render(child, depth + 1)
    return lines


provider = get_registry().get(PROVIDER)
engine = VerificationEngine(provider, estimator=estimator_for(provider))


async def finding_for(record_id: str) -> tuple[ClaimRecord, VerificationFinding]:
    """One record, and what the engine returns for it against the live chain."""
    path = next(CLAIMS.glob(f"{record_id}-*.toml"))
    record = load_record(path, corpus_dir=CORPUS)
    report = await engine.verify_post(record.post, Extraction(claims=(record.claim,)))
    return record, report.findings[0]


for record_id in ("0001", "0025", "0026"):
    # Top-level `await`, not `anyio.run`: a notebook kernel already has a running event loop,
    # and `anyio.run` refuses to start a second one in the same thread.
    record, finding = await finding_for(record_id)
    display(Markdown(f"### {record.id} — `{finding.verdict.value}`\n\n> {record.quote}"))
    if finding.reason:
        display(Markdown(f"**the engine's reason:** {finding.reason}"))
    display(Markdown("\n".join(render(derive_finding(finding).root))))

### 0001 — `unresolved`

> (a) 20,452.85228 BTC at address 3Pja5FPK1wFB9LkWWJai8XYL1qjbqqT9Ye

**the engine's reason:** the claim names a period that has ended, and this check reads the present balance; what an address held at a past height needs a provider that reports a holding as of one, which is not configured here

- **claim** — balance claim
  (a) 20,452.85228 BTC at address 3Pja5FPK1wFB9LkWWJai8XYL1qjbqqT9Ye
  - `addresses`: 3Pja5FPK1wFB9LkWWJai8XYL1qjbqqT9Ye
  - `amount_text`: 20,452.85228 BTC
  - `priced_elements`: amount
  - `provenance_strength`: printout
  - `txid`: None
  - `type`: balance
  - `window_end`: 2026-03-30T00:00:00+00:00
  - `window_start`: 2026-03-30T00:00:00+00:00
  - **verdict** — unresolved
    what the chain shows, which is a categorical finding and not a probability
    - `is_informative`: False
    - `method`: balance
    - **evidence** — the chain data
      every fact the finding rests on, each checkable on its own
      - `address`: 3Pja5FPK1wFB9LkWWJai8XYL1qjbqqT9Ye
      - `txids`: 
      - **evidence** — read from esplora-blockstream
        a fetch carries the provider's own say-so that it served this; a hand-supplied artifact does not, and the strength is recorded separately from the verdict
        - `endpoint`: None
        - `fetched_at`: 
        - `provenance_strength`: printout
        - `provider`: esplora-blockstream
    - **because** — the claim names a period that has ended, and this check reads the present balance; what an address held at a past height needs a provider that reports a holding as of one, which is not configured here
      the engine's own wording, reproduced rather than paraphrased — the refusal reasons are specific and substituting a summary would lose the distinction between them
      - `reason`: the claim names a period that has ended, and this check reads the present balance; what …
  - **assumption** — amount tolerance: amount stated exactly, with no hedge word and no stated tolerance
    - `assumption`: amount tolerance: amount stated exactly, with no hedge word and no stated tolerance

### 0025 — `contradicted`

> On or about and between May 31, 2019 and March 29, 2020, approximately 13,469 newly minted BTC were sent to address 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR (the “3QL Wallet”).

**the engine's reason:** the subject's 1578 movements in the window total 4023382140969, which is outside the claimed band around 1346900000000

- **claim** — transfer claim
  On or about and between May 31, 2019 and March 29, 2020, approximately 13,469 newly minted BTC were sent to address 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR (the “3QL Wallet”).
  - `addresses`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
  - `amount_text`: 13,469 BTC
  - `priced_elements`: amount
  - `provenance_strength`: printout
  - `txid`: None
  - `type`: transfer
  - `window_end`: 2020-03-29T23:59:59+00:00
  - `window_start`: 2019-05-31T00:00:00+00:00
  - **verdict** — contradicted
    what the chain shows, which is a categorical finding and not a probability
    - `is_informative`: True
    - `method`: transfer
    - **evidence** — 1578 movement(s), summed
      every fact the finding rests on, each checkable on its own
      - `claimed_nominal`: 1346900000000
      - `movements_in_window`: 1578
      - `observed_total`: 4023382140969
      - `txids`: 1408 items: 8c926c6d5fad992f2a2907506a4f0ec5f03e968622a92791c56729eecc60a78b, ad1fbbb6626d78bf3a5926a5dcc53a28b71bce81bdec0efda95c9fc2f9b6b2f4, 5d64276ffbdd4828a437e76a974b9858d64b76e545c8a3c384c3ac0f5d2371fd…
      - **evidence** — transfer 1259375159 in 8c926c6d5fad…
        the amount is the recorded value of this movement
        - `amount`: 1259375159
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 8c926c6d5fad992f2a2907506a4f0ec5f03e968622a92791c56729eecc60a78b
        - `via`: coinbase
      - **evidence** — transfer 609456065 in ad1fbbb6626d…
        the amount is the recorded value of this movement
        - `amount`: 609456065
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 1
        - `src`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `txid`: ad1fbbb6626d78bf3a5926a5dcc53a28b71bce81bdec0efda95c9fc2f9b6b2f4
        - `via`: utxo
      - **evidence** — transfer 1282859036 in 5d64276ffbdd…
        the amount is the recorded value of this movement
        - `amount`: 1282859036
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 5d64276ffbdd4828a437e76a974b9858d64b76e545c8a3c384c3ac0f5d2371fd
        - `via`: coinbase
      - **evidence** — transfer 1321874026 in 6403390f6a3c…
        the amount is the recorded value of this movement
        - `amount`: 1321874026
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 6403390f6a3c010b973c011e82f9a186a1300fd10b4e80c4a20e6bc06edaaa05
        - `via`: coinbase
      - **evidence** — transfer 3296827551 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 3296827551
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 33UB1zVipqDkzibpKKYFFLTpdw1uCRUG5V
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 660525857 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 660525857
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 3Js4ruBCwPxU74ZRZcsTosxyQ7yB1pmdHa
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 530567971 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 530567971
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 32NHy5m1A4ATQzrfyti9YWAJZs9gWPNF1S
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 316105 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 316105
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 3N7UWhYXR6Xb6dMDTdXJkS1NenbVcyK1i5
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 332319 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 332319
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 31wrujVVhf92puvwe4uE1cDgKNc4gXuQRz
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 5966578 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 5966578
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 37GiZxxWdtJBWWKTvTWDgRvSfPPFAsycpe
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 915694 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 915694
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 38V6NFCPZAsEy4tp8Lxx5CreXcQtQzXtoF
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 1648414 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 1648414
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 37a7R2BCyx3BoFp1VpaNJYxLzgAXYMmo1Z
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 2721235 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 2721235
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 3CCHGVZi4Q6h1tDRGfAkvFgBvLWaGVzJ5g
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 178276 in 320ce411cb8a…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 178276
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 5
        - `src`: 3JFCiaETgNLRD2Yu4rogdBYXWbwXY2SYJv
        - `txid`: 320ce411cb8ad7af147c4f610ca882bcbc49a6a960c757765bae450c2416a48c
        - `via`: utxo
      - **evidence** — transfer 134376293 in a7627642f529…
        the amount is the recorded value of this movement
        - `amount`: 134376293
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 1
        - `src`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `txid`: a7627642f52918d3a512269d492541c609115dfea828789ecc3e38ad4e30ca36
        - `via`: utxo
      - **evidence** — transfer 1281746513 in 3092d4eb606a…
        the amount is the recorded value of this movement
        - `amount`: 1281746513
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 3092d4eb606a067e934ebea967d8ccbf9a8da266f5bb8258a1b90bff2c9f6677
        - `via`: coinbase
      - **evidence** — transfer 1253427778 in 0f83fd36fa1d…
        the amount is the recorded value of this movement
        - `amount`: 1253427778
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 0f83fd36fa1db6b2b1fcf966aef0cadfeb67fa404bc90411d17087e6a5b7eff0
        - `via`: coinbase
      - **evidence** — transfer 1582736095 in 9235f8c27e6c…
        the amount is the recorded value of this movement
        - `amount`: 1582736095
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 1
        - `src`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `txid`: 9235f8c27e6c7a159bdb54d36795453d8425b3d41cc15265b47894e682624691
        - `via`: utxo
      - **evidence** — transfer 1268862254 in 8bc8509db9ff…
        the amount is the recorded value of this movement
        - `amount`: 1268862254
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 8bc8509db9fffe799195520d0e7382db36bb8d01583a23c13e0395859c6679a2
        - `via`: coinbase
      - **evidence** — transfer 1266519121 in 3710efb60f1d…
        the amount is the recorded value of this movement
        - `amount`: 1266519121
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 3710efb60f1d2824ab805afaf5e782aba4f4e862e22eac5a7a306ec5e9540aee
        - `via`: coinbase
      - **evidence** — transfer 1331146790 in 5d3f049e8c1e…
        the amount is the recorded value of this movement
        - `amount`: 1331146790
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 5d3f049e8c1eb00b94eca402767c01558472e95aab7e9b1677d0c9c85fcbb606
        - `via`: coinbase
      - **evidence** — transfer 1296460756 in de7f2ee41247…
        the amount is the recorded value of this movement
        - `amount`: 1296460756
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: de7f2ee4124788dfdcac61b73c6f58dee8c7ebd355d691f6d2f58220924ba3b7
        - `via`: coinbase
      - **evidence** — transfer 1274400110 in 1151a446fce0…
        the amount is the recorded value of this movement
        - `amount`: 1274400110
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 1151a446fce097e2f61ba4cc922c34f54faa294e09103ae5602c5df7db69bc67
        - `via`: coinbase
      - **evidence** — transfer 967962194 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 967962194
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 36YdyWq69aoPjFE7DZf5MQUc28SGVyRogP
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 400930 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 400930
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 3NtcCL5vkx81TW5mpuDQxgnP1uh7xccG4h
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 681848 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 681848
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 35nn3NfmGSi97Xqkj7XV6RutuspG4pu5Su
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 1070237 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 1070237
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 3HJjQPi9y35WBzu5YBiuYHwUFbz7wohoQk
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 11612981 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 11612981
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 35Lz7dqc7SLHGt7YxtfiwHnQxTvgYYCHFu
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 1618120 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 1618120
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 31oCi6qRw6HbRU2m3zG6MYx5p6uK67LmA4
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 339153 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 339153
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 3F5qdN2i5fwjSqPUfwo5gdcuAhHZpPevtx
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 696749 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 696749
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 324aTUbLBzXPQHMSftsyP2snTxcjhe5HnP
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 6327403 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 6327403
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 3NkRab3P2XDK6sAEUadGUn8PvssuFQRWf5
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 9290385 in 57df074b9a24…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 9290385
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: 3LvA4nuWVsEeJLigwwsyEgtd2wjaUsHUh2
        - `txid`: 57df074b9a244f3d70df91d8dfe0c8040011ceb225da0c081a0f387e64a4c23c
        - `via`: utxo
      - **evidence** — transfer 592249386 in 692c5c043d5f…
        the amount is the recorded value of this movement
        - `amount`: 592249386
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 1
        - `src`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `txid`: 692c5c043d5f931af7045c85c289a9c198afcb9814ae5ca8f0c5c937c50e8ff5
        - `via`: utxo
      - **evidence** — transfer 1315666546 in a0ad300808d8…
        the amount is the recorded value of this movement
        - `amount`: 1315666546
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: a0ad300808d879b76fea433a1c454b0008ec970ee8f11e409205f91637c15d3c
        - `via`: coinbase
      - **evidence** — transfer 1301294124 in ce3d42630cbc…
        the amount is the recorded value of this movement
        - `amount`: 1301294124
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: ce3d42630cbc43d06de43e4aa3f2a8defb20b1ecd13d743d9b04bfdd0fdeb77c
        - `via`: coinbase
      - **evidence** — transfer 1304062604 in 4ee600cce053…
        the amount is the recorded value of this movement
        - `amount`: 1304062604
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 4ee600cce053bcb0de6964a7d14d3267082f5e16af4adf9799c02ffb917d924b
        - `via`: coinbase
      - **evidence** — transfer 1323909286 in 4144bc8862c5…
        the amount is the recorded value of this movement
        - `amount`: 1323909286
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 4144bc8862c51a1c9ae0296188511fec6a64089677e541a1cb5cce7e24f325c6
        - `via`: coinbase
      - **evidence** — transfer 154329117 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 154329117
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 38e3EyQGq2q5WwmPMWs9bqbMpQuyj4sBK2
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 151793553 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 151793553
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 36EHjjQ2sh5KB1pcjXjtBYAaUjfnANavhs
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 146984748 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 146984748
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 3HdmHKjUQKUDt7HByt6iA6ph2VqhjujVTx
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 269780 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 269780
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 3DqZgjCcuzc2opjzUEiNAzStiVVC8scG68
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 1578375 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 1578375
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 3Fd2Ym4ZC8JoQC7h14REs4ti9HbNxysEWv
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 4518706 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 4518706
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 3A8yfAuVmUYUVzmAPGkW9qzxzcdcfTgxX5
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 241357 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 241357
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 3LNDp6QMSy1G6iAdDjUbgY6hbEYDLwMMXs
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 19607280 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 19607280
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 39HbQPyxR3M9GqHHV1UVMYGDfyK6rfP3EG
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 3520285 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 3520285
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 39ha6xn5UwjFCYytZ85EXfWnW7P1jbKyG1
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 17156799 in b91a2b8ac2c3…
        the amount is an apportioned share of the transaction's outputs, not a recorded value
        - `amount`: 17156799
        - `amount_basis`: apportioned (inferred)
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 2
        - `src`: 36F2spPQoRbR7WtbqqUN77dmcmZihoXTr8
        - `txid`: b91a2b8ac2c3a03e412ab574f46da640cbe7a2941199ea9d193527767b3300e2
        - `via`: utxo
      - **evidence** — transfer 411082165 in 858892e7a585…
        the amount is the recorded value of this movement
        - `amount`: 411082165
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 1
        - `src`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `txid`: 858892e7a5859fcfc9003e4a221f12c06def9f533385a857a3113b1ca6aef81a
        - `via`: utxo
      - **evidence** — transfer 1256660796 in 5661a6b5a14c…
        the amount is the recorded value of this movement
        - `amount`: 1256660796
        - `amount_basis`: recorded
        - `dst`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR
        - `index`: 0
        - `src`: None
        - `txid`: 5661a6b5a14c12fae736fb715b4d04da1acb1908ece8c6a5bde2adf6ec94d1a2
        - `via`: coinbase
      - **evidence** — read from esplora-blockstream
        a fetch carries the provider's own say-so that it served this; a hand-supplied artifact does not, and the strength is recorded separately from the verdict
        - `endpoint`: get_address_transactions
        - `fetched_at`: 2026-10-12T10:30:35.109127+00:00
        - `provenance_strength`: printout
        - `provider`: esplora-blockstream
      - **evidence** — the scan was exhaustive
        an exhaustive scan is what licenses the first proposition's probability of 1
        - `scan_complete`: True
        - `transfers_truncated`: True
    - **because** — the subject's 1578 movements in the window total 4023382140969, which is outside the claimed band around 1346900000000
      the engine's own wording, reproduced rather than paraphrased — the refusal reasons are specific and substituting a summary would lose the distinction between them
      - `reason`: the subject's 1578 movements in the window total 4023382140969, which is outside the cla…
  - **assumption** — amount tolerance: amount stated exactly, with no hedge word and no stated tolerance
    - `assumption`: amount tolerance: amount stated exactly, with no hedge word and no stated tolerance
  - **caveat** — a total is a sum over movements this library chose to add; a claim that meant one payment, and was recorded as a total, is refuted here by its own arithmetic
    - `caveat`: a total is a sum over movements this library chose to add; a claim that meant one paymen…

### 0026 — `contradicted`

> on or about and between July 18, 2019 and September 30, 2019, the 3QL Wallet sent approximately 2,138.4 BTC to the 364 Wallet.

**the engine's reason:** the subject's 0 movements to 364XjdP6Jbpm945DW4M6EobXLEmJv964rC in the window total 0, which is outside the claimed band around 213840000000

- **claim** — transfer claim
  on or about and between July 18, 2019 and September 30, 2019, the 3QL Wallet sent approximately 2,138.4 BTC to the 364 Wallet.
  - `addresses`: 3QLeXx1J9Tp3TBnQyHrhVxne9KqkAS9JSR, 364XjdP6Jbpm945DW4M6EobXLEmJv964rC
  - `amount_text`: 2,138.4 BTC
  - `priced_elements`: recipient, amount
  - `provenance_strength`: printout
  - `txid`: None
  - `type`: transfer
  - `window_end`: 2019-09-30T23:59:59+00:00
  - `window_start`: 2019-07-18T00:00:00+00:00
  - **verdict** — contradicted
    what the chain shows, which is a categorical finding and not a probability
    - `is_informative`: True
    - `method`: transfer
    - **evidence** — the chain data
      every fact the finding rests on, each checkable on its own
      - `claimed_nominal`: 213840000000
      - `movements_in_window`: 0
      - `observed_total`: 0
      - `txids`: 
      - **evidence** — read from esplora-blockstream
        a fetch carries the provider's own say-so that it served this; a hand-supplied artifact does not, and the strength is recorded separately from the verdict
        - `endpoint`: get_address_transactions
        - `fetched_at`: 2026-10-12T10:30:51.316123+00:00
        - `provenance_strength`: printout
        - `provider`: esplora-blockstream
      - **evidence** — the scan was exhaustive
        an exhaustive scan is what licenses the first proposition's probability of 1
        - `scan_complete`: True
        - `transfers_truncated`: False
    - **because** — the subject's 0 movements to 364XjdP6Jbpm945DW4M6EobXLEmJv964rC in the window total 0, which is outside the claimed band around 213840000000
      the engine's own wording, reproduced rather than paraphrased — the refusal reasons are specific and substituting a summary would lose the distinction between them
      - `reason`: the subject's 0 movements to 364XjdP6Jbpm945DW4M6EobXLEmJv964rC in the window total 0, w…
  - **assumption** — amount tolerance: amount stated exactly, with no hedge word and no stated tolerance
    - `assumption`: amount tolerance: amount stated exactly, with no hedge word and no stated tolerance
  - **caveat** — a total is a sum over movements this library chose to add; a claim that meant one payment, and was recorded as a total, is refuted here by its own arithmetic
    - `caveat`: a total is a sum over movements this library chose to add; a claim that meant one paymen…

## What the verifier changes moved

Three verdicts in this corpus would have been **different, and wrong**, before the changes the corpus
itself motivated:

| the filing says | before | now |
|---|---|---|
| a schedule row | contradicted — it holds nothing today | a gap: unobtainable |
| a receipt | checked on that address's spending | checked on what arrived |
| a period's figure | checked against each movement | summed, and reported |

None of them makes the corpus price anything. What a claim has to be for this library to say
`supported` is in `results/report.md`.
